In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import json

df = pd.read_csv("../data/processed/train_scaled.csv")

with open("../data/processed/top_sensors.json") as f:
    sensors = json.load(f)

In [2]:
from sklearn.model_selection import train_test_split

engine_ids = df["engine_id"].unique()

train_engines, test_engines = train_test_split(
    engine_ids, test_size=0.2, random_state=42
)

train_df = df[df["engine_id"].isin(train_engines)]
test_df = df[df["engine_id"].isin(test_engines)]

In [3]:
def create_sequences(df, sensors, window):

    X = []
    y = []

    for engine in df["engine_id"].unique():

        engine_df = df[df["engine_id"] == engine].sort_values("cycle")

        sensor_vals = engine_df[sensors].values
        rul_vals = engine_df["RUL"].values

        for i in range(len(engine_df) - window + 1):

            seq = sensor_vals[i:i+window]   # NO FLATTEN
            target = rul_vals[i+window-1]

            X.append(seq)
            y.append(target)

    return np.array(X), np.array(y)

In [5]:
window = 60

X_train, y_train = create_sequences(train_df, sensors, window)
X_test, y_test = create_sequences(test_df, sensors, window)

np.savez(
    "../data/processed/lstm_window_60.npz",
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test
)

print(X_train.shape)
print(X_test.shape)

(11841, 60, 10)
(2890, 60, 10)


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import mean_squared_error

In [5]:
data = np.load("../data/processed/lstm_window_60.npz")

X_train = torch.tensor(data["X_train"], dtype=torch.float32)
y_train = torch.tensor(data["y_train"], dtype=torch.float32).view(-1,1)

X_test = torch.tensor(data["X_test"], dtype=torch.float32)
y_test = torch.tensor(data["y_test"], dtype=torch.float32).view(-1,1)

In [6]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=10, hidden_size=128, num_layers=2):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )

        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)

        # take last timestep output
        out = out[:, -1, :]
        out = self.dropout(out)
        out = self.fc(out)
        return out

In [ ]:
class BiLSTMModel(nn.Module):
    def __init__(self, input_size=10, hidden_size=128, num_layers=2):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        # because bidirectional → hidden_size * 2
        self.fc = nn.Linear(hidden_size * 2, 1)

    def forward(self, x):
        out, _ = self.lstm(x)

        out = out[:, -1, :]   # last timestep
        out = self.fc(out)

        return out

In [ ]:
# model = LSTMModel()

model = BiLSTMModel()

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

In [8]:
epochs = 50
batch_size = 128

for epoch in range(epochs):

    perm = torch.randperm(X_train.size(0))
    total_loss = 0

    for i in range(0, X_train.size(0), batch_size):
        idx = perm[i:i+batch_size]

        xb = X_train[idx]
        yb = y_train[idx]

        pred = model(xb)
        loss = criterion(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / (X_train.size(0) // batch_size)

    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

Epoch 1, Loss: 6072.4821
Epoch 2, Loss: 4996.8461
Epoch 3, Loss: 4352.6615
Epoch 4, Loss: 3819.6099
Epoch 5, Loss: 3380.8561
Epoch 6, Loss: 3019.2122
Epoch 7, Loss: 2718.6717
Epoch 8, Loss: 2479.0291
Epoch 9, Loss: 2290.2118
Epoch 10, Loss: 2139.2290
Epoch 11, Loss: 2021.2325
Epoch 12, Loss: 1934.7685
Epoch 13, Loss: 1871.7750
Epoch 14, Loss: 1819.5702
Epoch 15, Loss: 1786.6289
Epoch 16, Loss: 1758.2714
Epoch 17, Loss: 1738.4568
Epoch 18, Loss: 1731.9364
Epoch 19, Loss: 1718.1042
Epoch 20, Loss: 1718.2255
Epoch 21, Loss: 1716.5924
Epoch 22, Loss: 1714.5335
Epoch 23, Loss: 1713.6269
Epoch 24, Loss: 1711.4198
Epoch 25, Loss: 1713.7676
Epoch 26, Loss: 1405.3490
Epoch 27, Loss: 757.3791
Epoch 28, Loss: 581.7908
Epoch 29, Loss: 468.5206
Epoch 30, Loss: 388.4635
Epoch 31, Loss: 324.5862
Epoch 32, Loss: 280.0788
Epoch 33, Loss: 244.9442
Epoch 34, Loss: 217.6295
Epoch 35, Loss: 193.9096
Epoch 36, Loss: 176.1366
Epoch 37, Loss: 162.7397
Epoch 38, Loss: 145.4847
Epoch 39, Loss: 144.8734
Epoch 40

In [9]:
model.eval()

with torch.no_grad():
    pred = model(X_test).numpy()

rmse = np.sqrt(mean_squared_error(y_test.numpy(), pred))

print("LSTM RMSE:", rmse)

LSTM RMSE: 13.0653324250458
